# Assignment 1: False Alarms

Author: Fredrik Boglind

## Introduction

Alarm fatigue is a critical problem in intensive care units (ICUs), where clinical staff are exposed to frequent monitor alarms — many of which are false positives. We use the **VTaC dataset** (Ventricular Tachycardia annotated alarms from ICUs) to build a classifier that predicts whether a ventricular tachycardia (VT) alarm is **True** or **False**.

**Dataset reference:**  
Lehman et al., *VTaC: A Benchmark Dataset of Ventricular Tachycardia Alarms from ICU Monitors*, NeurIPS 2023 Datasets and Benchmarks Track.  
Dataset available at [PhysioNet](https://physionet.org/content/vtac/1.0/) (DOI: 10.13026/z4f3-1f07).  
Associated GitHub: [github.com/ML-Health/VTaC](https://github.com/ML-Health/VTaC).

### Imports

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import os
from pathlib import Path
from scipy.signal import butter, filtfilt, iirnotch

import sklearn as sk
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
from sklearn.feature_selection import RFE

import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import skew, kurtosis

from utility_functions import (
    classify_signal, load_event, load_all_events,
    filter_all, normalize_per_sample, extract_features,
)

import warnings
warnings.filterwarnings('ignore')

## Data Loading

### Dataset structure

According to the VTaC data description (Lehman et al., 2023), the dataset is organised as follows:

- **`event_labels.csv`**: One row per alarm event. The `record` column identifies the patient recording, the `event` column identifies the specific alarm, and the `decision` column contains the expert-annotated label (True = real VT, False = false alarm).
- **`benchmark_data_split.csv`**: Assigns each event to a `train`, `val`, or `test` split. 

- **`waveforms/`**: A folder containing the actual physiological recordings in WFDB format. Sub-folders are named by patient record, and each contains up to five alarm event files.

We start by loading and merging the two CSV files into a unified dataframe with event identity, its split assignment, and its label.

In [ ]:
DATA_DIR = Path('raw_data/vtac-a-benchmark-dataset-of-ventricular-tachycardia-alarms-from-icu-monitors-1.0/')
WAVEFORM_DIR = DATA_DIR / 'waveforms'
# Load the metadata files
labels_df = pd.read_csv(DATA_DIR / 'event_labels.csv')
splits_df = pd.read_csv(DATA_DIR / 'benchmark_data_split.csv')

# Merge to get one row per event with its split + label
df = splits_df.merge(labels_df, on=['record', 'event'], how='inner')

# Shape of original dfs
print(f"Labels:  {labels_df.shape}")
print(f"Splits:  {splits_df.shape}")
# Shape of merged df
print(f"Merged:  {df.shape}")
print(f"\nColumns after merge:\n{df.columns.tolist()}")
df.head()

## Task 1: Predict whether or not an alarm is True or False

### Data Exploration

We explore the dataset in two stages: first the **metadata** (labels, splits, class balance), then the **waveform recordings** themselves. What we want is to understand what we're working with before making any preprocessing and modelling decisions.

#### 1.1 Class balance

In [ ]:
print(f"Total alarm events: {len(df)}")
print(f"Unique patient records: {df['record'].nunique()}")

print(f"\nLabel distribution")
print(df['decision'].value_counts())
true_pct = df['decision'].mean() * 100
print(f"\nClass balance: {true_pct:.1f}% True alarms, {100-true_pct:.1f}% False alarms")

The dataset contains 5,037 events — 28.6% true VT alarms and 71.4% false alarms. We can see that most VT alarms are false positives. We will need to account for this imbalance.

#### 1.2 Split distribution

In [ ]:
ct = pd.crosstab(df['split'], df['decision'], margins=True)
print(ct)

print("\nClass proportions per split:")
for split in ['train', 'val', 'test']:
    subset = df[df['split'] == split]
    print(f"  {split}: {subset['decision'].mean():.1%} True  (n={len(subset)})")

The benchmark split seems to be stratified, it has about the same class proportions (~28% True). The training set has 4060 events while the val and test sets have ~500 events each.

#### 1.3 Events per patient

In [ ]:
events_per_record = df.groupby('record')['event'].count()
print(events_per_record.describe())

fig, ax = plt.subplots(figsize=(8, 3))
events_per_record.value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_xlabel('Number of alarm events per patient record')
ax.set_ylabel('Count of patient records')
ax.set_title('Distribution of events per patient')
plt.tight_layout()
plt.show()

Most patients have between 1 and 5 events, meaning that some patients have more alarm events than others.

#### 1.4 Exploring the waveform files

Next let's look at the actual waveform data. According to the data description:

> *"Each waveform record in the current release consists of a 6-minute segment that encompasses the onset of the VT alarm. This segment includes 5 minutes of waveform data preceding the alarm onset and 1 minute following it. [...] Each waveform recording contains ECG leads and one or more pulsatile waveforms (photoplethysmogram and/or arterial blood pressure waveforms). All signals were uniformly resampled to 250 Hz."*  
> — Lehman et al., 2023

The files are stored in [WFDB format](https://physionet.org/content/wfdb/). Each event produces a pair of files (a [`.dat` binary file](https://physionet.org/physiotools/wpg/wpg_39.htm#Signal-Files) and a [`.hea` header file)](https://physionet.org/physiotools/wpg/wpg_39.htm#Header-Files). The `wfdb` Python library lets us read both.
We look at the folder structure for one patient, then open a single event to see what's inside.

##### Folder structure

In [ ]:
# Pick first patient record in the dataframe
example_record = df.iloc[0]['record']
record_dir = WAVEFORM_DIR / example_record

print(f"Patient record folder: {record_dir}")
print(f"Contents:")
for f in sorted(record_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

Each patient folder contains pairs of `.dat` and `.hea` files — one pair per alarm event. The `.hea` file is a small text header describing the recording (signal names, sampling rate, length, etc.), while the `.dat` file contains the actual signal data in binary.

##### Reading a single event

In [ ]:
example_event = df.iloc[0]['event']
example_path = str(WAVEFORM_DIR / example_record / example_event)
header = wfdb.rdheader(example_path)
header_keys=header.__dict__.keys()
print("\n".join(header_keys))

In the [WFDB documentation](https://www.physionet.org/physiotools/wag/header-5.htm) we can find an explanation of the headers.

The following attributes seem particulary interesting*:
`n_sig`: Number of Signals (not necessarily equal to the number of signal files, since two or more signals can share a signal file)
`sig_name`: Name of the signal, a list of channel names 
`fs`: Sampling Frequency
`sig_len`: Signal Length
`units`: Units

*In the VTaC [GitHub repo](https://github.com/ML-Health/VTaC) we can see which attributes were used

In [ ]:
# Read just the header first (lightweight — no signal data loaded)
header = wfdb.rdheader(example_path)

print(f"Event: {example_event}")
print(f"Number of signals: {header.n_sig}")
print(f"Signal names:      {header.sig_name}")
print(f"Sampling frequency: {header.fs} Hz")
print(f"Signal length:     {header.sig_len} samples")
print(f"Duration:          {header.sig_len / header.fs:.1f} seconds ({header.sig_len / header.fs / 60:.1f} minutes)")
print(f"Units:             {header.units}")

This confirms what the data description told us: a 6-minute recording (90,000 samples at 250 Hz). The signal names tell us which channels are available in this particular event. Next we load the actual signal values and look at the numbers.

In [ ]:
# Load the full record (header + signal data)
record = wfdb.rdrecord(example_path)

# The signal data is in record.p_signal, a 2D numpy array: (samples, channels)
print(f"Signal data shape: {record.p_signal.shape}")
print(f"  → {record.p_signal.shape[0]} time steps × {record.p_signal.shape[1]} channels")
print(f"\nSignal names: {record.sig_name}")
print(f"\nBasic statistics per channel:")
for i, name in enumerate(record.sig_name):
    col = record.p_signal[:, i]
    n_nan = np.isnan(col).sum()
    print(f"  {name:8s}  min={np.nanmin(col):10.3f}  max={np.nanmax(col):10.3f}  "
          f"mean={np.nanmean(col):10.3f}  NaNs={n_nan}")

The signal values are in their original physical units (e.g., millivolts for ECG, mmHg for ABP). The ranges differ between channel types. ECG values are small (sub-millivolt) while ABP values are in the tens to hundreds. We will need to do normalization later.

##### Plotting a single event

In [ ]:
# Plot all channels for the one event
fig, axes = plt.subplots(len(record.sig_name), 1, figsize=(14, 2.5 * len(record.sig_name)), sharex=True)
if len(record.sig_name) == 1:
    axes = [axes]

t = np.arange(record.sig_len) / record.fs  # time axis in seconds

for i, (ax, name) in enumerate(zip(axes, record.sig_name)):
    ax.plot(t, record.p_signal[:, i], linewidth=0.4)
    ax.set_ylabel(f"{name} ({record.units[i]})")
    ax.grid(True, alpha=0.3)
    # Mark the alarm onset at t=300s (5 minutes)
    ax.axvline(x=300, color='red', linestyle='--', linewidth=1, alpha=0.7)

axes[-1].set_xlabel('Time (seconds)')
label = df.iloc[0]['decision']
fig.suptitle(f"Event: {example_event} — {'TRUE alarm' if label else 'FALSE alarm'}\n"
             f"Red dashed line = alarm onset (5 min mark)", fontsize=12)
plt.tight_layout()
plt.show()

The red dashed line marks the alarm onset at the 5-minute mark. Everything to the left is the lead-up to the alarm; the final minute is what followed. This is important context for feature extraction later, we may want to focus on the region around the alarm onset rather than the full 6 minutes.

##### Comparing a True vs False alarm

In [ ]:
# Pick one True and one False alarm
true_row = df[df['decision'] == True].iloc[0]
false_row = df[df['decision'] == False].iloc[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, row, color in zip(axes, [true_row, false_row], ['tab:red', 'tab:blue']):
    rec = wfdb.rdrecord(str(WAVEFORM_DIR / row['record'] / row['event']))
    t = np.arange(rec.sig_len) / rec.fs
    # Plot the first ECG lead
    ax.plot(t, rec.p_signal[:, 0], linewidth=0.4, color=color)
    ax.axvline(x=300, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    label_str = 'TRUE alarm' if row['decision'] else 'FALSE alarm'
    ax.set_title(f"{row['event']} — {label_str} (showing: {rec.sig_name[0]})", fontsize=11)
    ax.set_ylabel(f"{rec.sig_name[0]} ({rec.units[0]})")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

Visually, we should see a difference, true VT alarms tend to show rapid arrythmia, while false alarms may show artifacts, noise, or normal rhythms that were misclassified by the monitor (Lehman et al). In these particular samples a difference seem to be discernible: The false alarm appears to be noisy, while the True alarm show that something is happening at 50s and ~300s. However, these are just two random samples and it is also worth noting that these two events have different first ECG leads (I vs II), we should therefore be careful when comparing them.

#### 1.5 Signal availability across the full dataset

So far we've looked at individual events. Now we need to understand the full picture: which signal types are present for all of the 5,037 events? Not every patient has every type of sensor attached, some events may lack ABP or PLETH recordings. We can read just the header of each event file (this is fast, since we don't need to load the signal data) and catalogue what's available.

In [ ]:
# Scan all event headers to catalogue signal availability
signal_info = []

for _, row in df.iterrows():
    path = str(WAVEFORM_DIR / row['record'] / row['event'])
    try:
        hdr = wfdb.rdheader(path)
        sig_upper = [s.upper() for s in hdr.sig_name]
        signal_info.append({
            'event': row['event'],
            'record': row['record'],
            'n_signals': hdr.n_sig,
            'sig_names': hdr.sig_name,
            'sig_len': hdr.sig_len,
            'has_pleth': 'PLETH' in sig_upper,
            'has_abp': 'ABP' in sig_upper,
            'n_ecg': sum(1 for s in sig_upper if s not in ('PLETH', 'ABP')),
        })
    except Exception as e:
        signal_info.append({
            'event': row['event'], 'record': row['record'],
            'n_signals': 0, 'sig_names': [], 'sig_len': 0,
            'has_pleth': False, 'has_abp': False, 'n_ecg': 0,
        })
        print(f"  WARNING: Could not read {row['event']}: {e}")

sig_df = pd.DataFrame(signal_info)
print(f"Scanned {len(sig_df)} event headers.")

In [ ]:
# Summarise signal availability
print("=== Signal availability ===")
print(f"  Events with PLETH: {sig_df['has_pleth'].sum()}/{len(sig_df)} ({sig_df['has_pleth'].mean():.1%})")
print(f"  Events with ABP:   {sig_df['has_abp'].sum()}/{len(sig_df)} ({sig_df['has_abp'].mean():.1%})")

print(f"\n=== Number of ECG leads per event ===")
print(sig_df['n_ecg'].value_counts().sort_index().to_string())

print(f"\n=== Total signals per event ===")
print(sig_df['n_signals'].value_counts().sort_index().to_string())

In [ ]:
# Find the event(s) with only 1 signal
one_sig = sig_df[sig_df['n_signals'] == 1]
print(f"Events with 1 signal: {len(one_sig)}")
for _, row in one_sig.iterrows():
    rec = wfdb.rdrecord(str(WAVEFORM_DIR / row['record'] / row['event']))
    print(f"\n  Event:   {row['event']}")
    print(f"  Signals: {rec.sig_name}")
    print(f"  Units:   {rec.units}")
    print(f"  Length:  {rec.sig_len} samples ({rec.sig_len/250:.0f}s)")
    print(f"  Stats:   min={rec.p_signal[:,0].min():.3f}  max={rec.p_signal[:,0].max():.3f}")
    print(f"  NaNs:    {np.isnan(rec.p_signal).sum()}")
    label = df[df['event'] == row['event']]['decision'].values[0]
    print(f"  Label:   {label}")

One event stood out since it contained only a single ECG, still, it seems to be a valid recording and it is therefore retained in the dataset. Three of its four channels will be filled with zeroes.

Next we look at what combination of signal names occur in the data.

In [ ]:
# What are the specific signal name combinations?
sig_df['combinations'] = sig_df['sig_names'].apply(lambda names: ' + '.join(sorted([s.upper() for s in names])))
print("Signal combinations (top 10)")
print(sig_df['combinations'].value_counts().head(10).to_string())

The most common recording setup includes 2–3 ECG leads plus PLETH (with or without ABP). Some events have as many as 12 ECG leads from full 12-lead recordings. ABP is absent in about 2/3 of all events.

In [ ]:
# How does signal length vary?
print("=== Signal length (samples) ===")
print(sig_df['sig_len'].describe())
print(f"\nDuration range: {sig_df['sig_len'].min()/250:.1f}s – {sig_df['sig_len'].max()/250:.1f}s")

n_standard = (sig_df['sig_len'] == 90000).sum()
print(f"Events with exactly 90000 samples (6 min): {n_standard}/{len(sig_df)} ({n_standard/len(sig_df):.1%})")

All 5,037 events are exactly 90,000 samples (6 minutes at 250 Hz), we knew this from the data description, but now we have confirmed it. No padding or truncation is needed.

**Key findings:**
- ECG lead count varies widely: most events have 3–4 ECG leads, but one event has only 1 and 71 events have the full 12. The most common combinations include leads II, V, and aVR.
- PLETH is available in 92.9% of events. ABP is available in only 36.5%, making it the primary source of missing data.
- One event (569ea7_0074) contains only a single ECG lead.
- All 5,037 events are exactly 90,000 samples (6 minutes at 250 Hz). No padding etc is needed.
- For our preprocessing pipeline, we will standardise to a fixed 4-channel layout: two ECG leads, PLETH, and ABP. This follows the approach used in the VTaC benchmark code (Lehman et al., 2023). This decision means discarding additional ECG leads in signal-rich events — a trade-off between information and consistency.

#### 1.6 Missing data and signal quality

It is worth checking whether signal availability correlates with the alarm label. If, for example, ABP is more often present in true alarm cases, then simply knowing whether ABP exists could leak information about the label. Conversely, if missingness is balanced, we can treat it as independent of the target.

In [ ]:
# Check missingness in relation to label
sig_merged = sig_df.merge(df[['event', 'decision']], on='event')

print("=== ABP availability by label ===")
print(pd.crosstab(sig_merged['decision'], sig_merged['has_abp'], margins=True))

print("\n=== PLETH availability by label ===")
print(pd.crosstab(sig_merged['decision'], sig_merged['has_pleth'], margins=True))

ABP availability is proportional across True and False alarms (~35% in both classes), and PLETH is similar too. It seems missingness doesn't strongly predict the label, meaning imputation won't introduce bias.

In [ ]:
# We sample a few events and check for NaN values
nan_counts = []
sample_idxs = np.random.RandomState(42).choice(len(df), size=min(200, len(df)), replace=False)

for idx in sample_idxs:
    row = df.iloc[idx]
    try:
        rec = wfdb.rdrecord(str(WAVEFORM_DIR / row['record'] / row['event']))
        n_nan = np.isnan(rec.p_signal).sum()
        nan_counts.append({'event': row['event'], 'n_nan': n_nan, 'total': rec.p_signal.size})
    except:
        pass

nan_df = pd.DataFrame(nan_counts)
n_with_nans = (nan_df['n_nan'] > 0).sum()
print(f"Sampled {len(nan_df)} events for NaN check:")
print(f"Events with NaN values: {n_with_nans}/{len(nan_df)}")
if n_with_nans > 0:
    print(f"Max NaN count in a single event: {nan_df['n_nan'].max()}")
else:
    print("No NaN values found in the sampled events.")

In [ ]:
# Investigate NaN patterns more closely
nan_details = []
sample_idxs = np.random.RandomState(42).choice(len(df), size=min(200, len(df)), replace=False)

for idx in sample_idxs:
    row = df.iloc[idx]
    try:
        rec = wfdb.rdrecord(str(WAVEFORM_DIR / row['record'] / row['event']))
        per_channel = [np.isnan(rec.p_signal[:, ch]).sum() for ch in range(rec.n_sig)]
        if any(n > 0 for n in per_channel):
            nan_details.append({
                'event': row['event'],
                'sig_names': rec.sig_name,
                'nans_per_channel': dict(zip(rec.sig_name, per_channel)),
                'total_samples': rec.sig_len,
            })
    except:
        pass

print(f"Events with NaNs: {len(nan_details)}/200")
for d in nan_details[:5]:  # show first 5
    print(f"\n  {d['event']}:")
    for name, count in d['nans_per_channel'].items():
        if count > 0:
            pct = count / d['total_samples'] * 100
            print(f"    {name}: {count}/{d['total_samples']} NaN ({pct:.0f}%)")

A sample of 200 events reveals that 12.5% have NaN values in one or more channels. These range from single-sample dropouts (sensor glitches?) to near-complete channel loss. This is distinct from the missing-channel problem, these signals exist in the file but have gaps. We handle NaNs by replacing them with zero (np.nan_to_num) while loading, before filtering. The brief glitches are small enough that filtering will smooth over them, while heavily NaN-filled channels will behave similarly to absent channels.

### Data Cleaning & Preprocessing

Based on our exploration, we now define a preprocessing pipeline. The key decisions are:

1. **Channel alignment** — Standardise all events to a fixed 4-channel layout: [ECG1, ECG2, PLETH, ABP]. Missing channels are zero-filled, and we track which channels are present.
2. **Signal filtering** — Remove noise using channel-specific filters (adapted from the [VTaC repository](https://github.com/ML-Health/VTaC)).
3. **Normalisation** — Z-score standardisation per sample.

Each step is documented and motivated below.

#### Step 1: Load waveforms into a consistent format

Different events have different numbers and names of signals. To process them uniformly, we map each signal to one of four standard slots based on its name. ECG signals (whatever their lead name) go into slots 0 and 1; PLETH goes into slot 2; ABP into slot 3. If a signal type is absent, that slot stays filled with zeros.

This zero-filling approach is a deliberate choice — it preserves all events in the dataset rather than discarding any. We track availability separately so that we can (a) avoid filtering/normalising empty channels, and (b) later compare models trained with vs. without imputation, as required by the assignment.

The `classify_signal`, `load_event`, and `load_all_events` functions are defined in `utility_functions.py`.

In [ ]:
SAMPLING_FREQ = 250
CHANNEL_NAMES = ['ECG1', 'ECG2', 'PLETH', 'ABP']

# Signal names that indicate ECG leads (the dataset uses various lead names)
ECG_LEAD_NAMES = {'II', 'I', 'III', 'V', 'AVR', 'AVL', 'AVF', 'MCL', 'ECG'}
PLETH_NAMES = {'PLETH'}
ABP_NAMES = {'ABP'}

# Quick test of the loading function
test_row = df.iloc[0]
wf, avail = load_event(test_row['event'], test_row['record'], WAVEFORM_DIR,
                        ECG_LEAD_NAMES, PLETH_NAMES, ABP_NAMES)
print(f"Shape: {wf.shape}")
print(f"Available: { {CHANNEL_NAMES[i]: avail[i] for i in range(4)} }")

#### Step 2: Load all events

In [ ]:
print("Loading all waveforms...")
waveforms, labels, availability, event_ids = load_all_events(df, WAVEFORM_DIR)
print(f"Done. Loaded {waveforms.shape[0]} events, shape: {waveforms.shape}")

In [ ]:
# Document what we have
print("=== Channel availability in loaded data ===")
for i, name in enumerate(CHANNEL_NAMES):
    present = availability[:, i].sum()
    missing = len(waveforms) - present
    print(f"  {name:6s}: {present} present, {missing} missing ({missing/len(waveforms):.1%})")

n_complete = (availability.sum(axis=1) == 4).sum()
print(f"\nEvents with all 4 channels: {n_complete}/{len(waveforms)} ({n_complete/len(waveforms):.1%})")
print(f"Events with missing channels: {len(waveforms) - n_complete}")
print(f"\n→ No events were deleted. Missing channels are zero-filled and tracked via the availability mask.")

Only 35% of events have all four channels available. The missing data is dominated by ABP (absent in 63.5% of events), with smaller amounts of missing PLETH (7.1%) and ECG2 (0.9%). ECG1 is always present. No events were deleted. Missing channels remain zero-filled and tracked. The impact of this missing data will be evaluated after feature extraction, where we compare imputation strategies against the zero-fill baseline (assignment requirement 5).

#### Step 3: Signal filtering

In order to remove noise we apply channel-specific filters, following the approach from the VTaC [GitHub repo](https://github.com/ML-Health/VTaC) (Lehman et al., 2023):

**ECG (leads 1 & 2):**
- 1 Hz highpass — removes slow baseline drift caused by breathing or electrode movement
- 30 Hz lowpass — removes high-frequency noise above the clinically relevant ECG range
- 60 Hz notch — removes power line interference (US standard)

**PLETH (PPG):**
- 60 Hz notch — removes power line interference
- 0.5–5 Hz bandpass — isolates the pulse waveform, removing both slow drift and high-freq noise

**ABP (arterial blood pressure):**
- 60 Hz notch — removes power line interference
- 16 Hz lowpass — smooths high-frequency artifacts while preserving the blood pressure waveform

All filters use low order (order 2) Butterworth design to minimise ringing. We only filter channels that are actually present — zero-filled missing channels are left untouched.

Filter code adapted from [filtering.py](https://github.com/ML-Health/VTaC) (Lehman et al., 2023). See `utility_functions.py` for the implementation.

In [ ]:
print("Applying signal filters...")
waveforms_filtered = filter_all(waveforms, availability)
print("Filtering complete.")

In [ ]:
# Visualise the effect of filtering on one event
example_idx = 0
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
t = np.arange(waveforms.shape[2]) / SAMPLING_FREQ

for ch, (ax, name) in enumerate(zip(axes, CHANNEL_NAMES)):
    if availability[example_idx, ch]:
        ax.plot(t, waveforms[example_idx, ch], alpha=0.4, linewidth=0.4, label='Raw', color='gray')
        ax.plot(t, waveforms_filtered[example_idx, ch], linewidth=0.5, label='Filtered')
        ax.legend(loc='upper right', fontsize=8)
    else:
        ax.text(0.5, 0.5, f'{name}: not available', transform=ax.transAxes,
                ha='center', va='center', fontsize=11, color='gray')
    ax.set_ylabel(name)
    ax.axvline(x=300, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (seconds)')
fig.suptitle(f'Raw vs Filtered — {event_ids[example_idx]}', fontsize=12)
plt.tight_layout()
plt.show()

Filtering reduces high-frequency noise in the ECG leads and isolates the pulse waveform in PLETH. The ABP channel is not present for this event.

#### Step 4: Normalisation

We apply **per-sample z-score normalisation**: each channel of each event is standardised using its own mean and standard deviation. This is preferred over population-level normalisation because:

- Signal amplitudes vary between patients (different body compositions, electrode placements)
- Different hospitals and monitor manufacturers produce different absolute values
- Per-sample normalisation makes each event self-contained

Only channels that are actually present are normalised. Zero-filled missing channels remain at zero.

*Normalisation approach from [VTaC preprocessing/standardize.py](https://github.com/ML-Health/VTaC).* See `utility_functions.py` for the implementation.

In [ ]:
print("Normalising...")
waveforms_normed = normalize_per_sample(waveforms_filtered, availability)
print("Done.")
print(f"Final waveform shape: {waveforms_normed.shape}")

#### Preprocessing summary

In [ ]:
# Reconstruct split assignments for the loaded events
event_to_split = df.set_index('event')['split'].to_dict()
split_labels = np.array([event_to_split[e] for e in event_ids])

print("Data after preprocessing")
for split_name in ['train', 'val', 'test']:
    mask = split_labels == split_name
    n = mask.sum()
    n_true = labels[mask].sum()
    print(f"  {split_name:5s}: {n} events  ({n_true} True, {n - n_true} False, {n_true/n:.1%} True)")

print(f"\nPipeline:")
print(f"  1. Loaded {len(waveforms)} events into 4-channel arrays (zero-fill missing signals)")
print(f"  2. Applied signal-specific filtering (ECG: HP+LP+notch, PLETH: bandpass, ABP: LP+notch)")
print(f"  3. Per-sample z-score normalisation")
print(f"  Output shape: {waveforms_normed.shape} — (events, channels, time samples)")
print(f"\nNo rows deleted. Missing channels tracked via availability mask ({(~availability).any(axis=1).sum()} events have one or more missing channels).")

All 5037 events were loaded into a 4 by 90,000 array with zero-filling for missing channels and nan_to_num for in-signal NaN values. Signal-specific filters were applied only to present channels. Per-sample z-score normalisation standardised each channel independently. No events were deleted. The class balance was preserved for all splits. 3,276 events (65%) have at least one missing channel (most of which are ABP).

### Feature Extraction

Next we extract statistical features from the preprocessed waveforms for model training. For each of the four channels in each event, we compute seven summary statistics: mean, standard deviation, minimum, maximum, skewness, kurtosis, and range. This produces a feature vector of 28 values per event (7 features × 4 channels).

Missing channels produce NaN values in their derived features, which will be handled by imputation in the next step. See `utility_functions.py` for the `extract_features` implementation.

In [ ]:
print("Extracting features...")
feat_df = extract_features(waveforms_normed, availability, CHANNEL_NAMES)
feat_df['decision'] = labels
feat_df['split'] = [event_to_split[e] for e in event_ids]

print(f"Feature matrix shape: {feat_df.shape}")
print(f"Features per event: {feat_df.shape[1] - 2}")  # minus label and split
print(f"\nNaN counts per feature:")
print(feat_df.drop(columns=['decision', 'split']).isnull().sum().to_string())

Skew and kurtosis features show slightly more NaN values than other features for each channel. This occurs when a present signal is constant or near-constant, causing the computation to be undefined. These are treated as additional missing values during imputation.

### Step 5: Imputation

We compare four strategies for handling missing values (following the methods in Lab 2), plus a fifth baseline of dropping incomplete rows. The imputers are fit on the training set only and then applied to validation and test sets, to prevent data leakage.

In [ ]:
# Separate features, labels, and splits
feature_cols = [c for c in feat_df.columns if c not in ['decision', 'split']]
X = feat_df[feature_cols]
y = feat_df['decision'].astype(int)
splits = feat_df['split']

# Split into train/val/test
X_train = X[splits == 'train'].copy()
X_val = X[splits == 'val'].copy()
X_test = X[splits == 'test'].copy()
y_train = y[splits == 'train']
y_val = y[splits == 'val']
y_test = y[splits == 'test']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"NaN values in train: {X_train.isnull().sum().sum()}")

Our chosen imputation methods are Zero-fill (following Lehman), Mean imputation, Median imputation and Iterative imputation.

In [ ]:
# Define imputation strategies
imp_methods = {}

# 1. Zero fill (as in VTaC repo)
imp_methods['zero_fill'] = SimpleImputer(strategy='constant', fill_value=0)

# 2. Mean imputation
imp_methods['mean'] = SimpleImputer(strategy='mean')

# 3. Median imputation
imp_methods['median'] = SimpleImputer(strategy='median')

# 4. Iterative imputation
imp_methods['iterative'] = IterativeImputer(random_state=42, max_iter=10)

# Apply each strategy: fit on train, transform train/val/test
imputed_datasets = {}

In [ ]:
for name, imputer in imp_methods.items():
    X_tr = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols, index=X_train.index)
    X_v = pd.DataFrame(imputer.transform(X_val), columns=feature_cols, index=X_val.index)
    X_te = pd.DataFrame(imputer.transform(X_test), columns=feature_cols, index=X_test.index)
    imputed_datasets[name] = (X_tr, X_v, X_te)
    print(f"\n{name}: NaNs remaining — train: {X_tr.isnull().sum().sum()}, val: {X_v.isnull().sum().sum()}")

# Drop rows with missing values
X_train_drop = X_train.dropna()
y_train_drop = y_train.loc[X_train_drop.index]
X_val_drop = X_val.dropna()
y_val_drop = y_val.loc[X_val_drop.index]
X_test_drop = X_test.dropna()
y_test_drop = y_test.loc[X_test_drop.index]

print(f"\ndrop_rows: Train {X_train.shape[0]} → {X_train_drop.shape[0]}, "
      f"Val {X_val.shape[0]} → {X_val_drop.shape[0]}, "
      f"Test {X_test.shape[0]} → {X_test_drop.shape[0]}")

### Step 6: Modelling

We evaluate two models:
- **Logistic Regression**: a linear model that serves as a simple, interpretable baseline. It directly outputs class probabilities, and its coefficients can be inspected to understand which features drive predictions. It also supports class_weight='balanced' to handle the class imbalance (Pedregosa et al., *Scikit-learn: Machine Learning in Python*, JMLR 2011).

- **Random Forest**: a non-linear ensemble method that can capture interactions between features (e.g., the combination of high ECG variability and low ABP variability) that a linear model would miss. It is robust to feature scaling and provides built-in feature importance estimates (Breiman, *Random Forests*, Machine Learning 2001).

We chose one linear and one non-linear model to assess whether the relationship between features and alarm outcomes is primarily linear or requires more complex modelling. Both models use `class_weight='balanced'` to upweight the minority class (True alarms at ~29%), which adjusts the loss function to penalise misclassification of true alarms more heavily.

In [ ]:
# Define models. We set class_weight='balanced' because of the 71/29 imbalance
models = {
    'logistic_regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'random_forest': RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
}

# We evaluate all the combinations of imputation strategy × model
results = []

for imp_name, (X_tr, X_v, X_te) in imputed_datasets.items():
    for model_name, model in models.items():
        clf = model.__class__(**model.get_params())  # fresh copy
        clf.fit(X_tr, y_train)
        y_pred = clf.predict(X_te)
        
        results.append({
            'imputation': imp_name,
            'model': model_name,
            'accuracy': accuracy_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
        })

# Evaluate drop_rows separately (different y sets)
for model_name, model in models.items():
    clf = model.__class__(**model.get_params())
    clf.fit(X_train_drop, y_train_drop)
    y_pred = clf.predict(X_test_drop)
    
    results.append({
        'imputation': 'drop_rows',
        'model': model_name,
        'accuracy': accuracy_score(y_test_drop, y_pred),
        'f1': f1_score(y_test_drop, y_pred),
        'precision': precision_score(y_test_drop, y_pred),
        'recall': recall_score(y_test_drop, y_pred),
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values('f1', ascending=False).to_string(index=False))

Iterative imputation produces the best F1 score (0.599) with logistic regression, though the differences between imputation strategies are small. Zero-fill consistently performs worst, suggesting that explicit imputation, even simple mean/median, provides a modest benefit over treating missing channels as zero. Dropping incomplete rows does not catastrophically hurt logistic regression performance despite reducing training data by 66%, but it does degrade random forest performance and makes test results non-comparable since they are evaluated on a different (smaller) subset.

### Step 7: Feature Importance

We select the best model — iterative imputation with logistic regression (F1 = 0.599) — and use **Recursive Feature Elimination (RFE)** to rank all 28 features by importance.

RFE works by repeatedly fitting the model, removing the least important feature at each step, and ranking features by the order in which they were eliminated. A ranking of 1 means the feature was the last to be removed (most important).

In [ ]:
# Use iterative imputation + logistic regression (best F1: 0.599)
X_tr, X_v, X_te = imputed_datasets['iterative']

# Fit the model
clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf.fit(X_tr, y_train)

# Recursive Feature Elimination
rfe = RFE(estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
          n_features_to_select=1, step=1)
rfe.fit(X_tr, y_train)

rfe_df = pd.DataFrame({
    'feature': feature_cols,
    'ranking': rfe.ranking_
}).sort_values('ranking')

print("Recursive Feature Elimination ranking")
print("(1 = last to be eliminated = most important)")
print(rfe_df.to_string(index=False))

# Visualise RFE ranking. Color represents signal type (ECG, PLETH, ABP).
fig, ax = plt.subplots(figsize=(10, 6))
rfe_sorted = rfe_df.sort_values('ranking')
colors = ['tab:red' if 'ECG' in f else 'tab:green' if 'PLETH' in f else 'tab:blue' for f in rfe_sorted['feature']]
ax.barh(rfe_sorted['feature'], rfe_sorted['ranking'], color=colors)
ax.set_xlabel('RFE Ranking (1 = most important)')
ax.set_title('Recursive Feature Elimination: Feature Rankings')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

RFE reveals that signal variability (standard deviation) is the most predictive feature type across all channels, with ABP_std ranking first. This suggests that true VT alarms produce characteristic changes in signal variability. Mean features are uninformative due to per-sample z-score normalisation. ABP features are highly informative when present, despite being missing in 63% of events — this helps explain why imputation strategy matters and why dropping incomplete rows hurts performance.

## Results and Discussion

### Summary of results

Our best model is **iterative imputation + logistic regression**, achieving an F1 score of **0.599** on the held-out test set (precision = 0.444, recall = 0.920, accuracy = 0.614). The high recall means the model correctly identifies 92% of true VT alarms, at the cost of relatively low precision — about 56% of the alarms it flags as true are actually false positives.

### Model comparison

Logistic regression consistently achieves higher recall (88–92%) than random forest (39–45%) across all imputation strategies, while random forest achieves higher precision (60–63%). In a clinical setting where missing a true VT alarm is dangerous, the high-recall logistic regression model is arguably more appropriate.

The fact that a simple linear model outperforms the random forest on F1 suggests that the relationship between our statistical features and alarm outcomes is largely linear. The random forest's lower recall indicates it is more conservative in predicting true alarms, possibly overfitting to majority-class patterns despite the balanced class weighting.

### Impact of imputation strategy

Iterative imputation produces the best F1, though the margin over mean and median imputation is small. All three outperform zero-fill, which confirms that informed imputation adds value over treating missing channels as zero. Dropping incomplete rows performs surprisingly well for logistic regression (F1 = 0.584) despite losing 66% of training data, but the results are not directly comparable since the test set is also reduced.

The importance of ABP features (ranked #1 by RFE despite 63% missingness) helps explain why imputation strategy matters: how we fill in missing ABP values directly affects the model's access to its most informative signal.

### Feature importance

RFE reveals that standard deviation is the most predictive feature type, with ABP_std, ECG2_std, and PLETH_std all ranking in the top 5. This makes sense physiologically: ventricular tachycardia produces a rapid, regular rhythm that changes the variability profile of physiological signals. Mean features rank last because per-sample z-score normalisation centres each channel at zero.

### Limitations

The statistical features we extract (mean, std, min, max, skew, kurtosis, range) are simple global summaries that ignore the temporal structure of the waveforms. More sophisticated approaches — such as frequency-domain features, morphological features (e.g. R-peak detection), or deep learning on the raw waveforms — could improve performance. The VTaC benchmark paper (Lehman et al., 2023) reports substantially better results with neural network architectures that operate on raw or minimally processed waveforms.

Additionally, our 4-channel layout discards ECG leads beyond the first two, which means we lose information in signal-rich events. A more flexible architecture could use all available leads.

### References

- Lehman et al., *VTaC: A Benchmark Dataset of Ventricular Tachycardia Alarms from ICU Monitors*, NeurIPS 2023 Datasets and Benchmarks Track. [PhysioNet](https://physionet.org/content/vtac/1.0/) (DOI: 10.13026/z4f3-1f07). [GitHub](https://github.com/ML-Health/VTaC).
- Pedregosa et al., *Scikit-learn: Machine Learning in Python*, JMLR 2011.
- Breiman, *Random Forests*, Machine Learning 2001.